# SNP Calling Pipeline: A Practical Hands-On Workflow

## Tier 3 - Applied Bioinformatics

This notebook walks through a complete, real-world SNP calling pipeline: from raw FASTQ reads to a final annotated variant report. We cover each tool, why it is used, what it produces, and how to interpret the results.

The pipeline is based on: https://github.com/Pavel-Kravchenko/snp-calling-pipeline

### Learning Objectives
- Understand the end-to-end SNP calling workflow
- Apply Trimmomatic for quality-based read trimming
- Align reads with HISAT2 using genomic DNA settings
- Process BAM files and call variants with samtools / bcftools
- Annotate variants with ANNOVAR against RefGene, dbSNP, 1000 Genomes, GWAS Catalog, and ClinVar
- Parse and summarise ANNOVAR output in Python

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Variant Calling and SNP Analysis](01_variant_calling_and_snp_analysis.ipynb) | [Next: RNA-seq Analysis: From Reads to Differential Expression →](../03_RNA_seq_Analysis/01_rna_seq_analysis.ipynb)

---

## 1. Pipeline Architecture

The pipeline accepts single-end FASTQ reads and produces a variant report annotated against multiple databases.

```
Raw FASTQ reads
       |
       v
  Trimmomatic  (quality trimming: TRAILING:20, MINLEN:50)
       |
       v
  HISAT2       (alignment to reference genome)
               --no-spliced-alignment --no-softclip
       |
       v
  samtools     SAM → BAM → sort → index → idxstats → depth
       |
       v
  samtools mpileup
       |
       v
  bcftools call -cv
       |        raw VCF
       v
  ANNOVAR convert2annovar.pl   (VCF → avinput)
       |
       v
  ANNOVAR annotate_variation.pl
       ├── dbSNP138    (flag known SNPs)
       ├── refGene     (gene/exon annotation)
       ├── 1000G       (population allele frequency)
       ├── GWAS Catalog (trait associations)
       └── ClinVar     (clinical significance)
       |
       v
  Final annotated report
```

### Pipeline File Structure

```
snp-calling-pipeline/
├── reads/                 ← put your .fastq files here
├── Human/                 ← HISAT2 index + reference FASTA per chromosome
│   ├── chr1.fasta
│   ├── chr1.1.ht2 ... chr1.8.ht2   ← HISAT2 index files
│   └── builder.sh         ← builds the HISAT2 index
├── annovar/
│   ├── humandb/           ← ANNOVAR databases (downloaded separately)
│   ├── convert2annovar.pl
│   └── annotate_variation.pl
├── Trimmomatic-0.36/
│   └── trimmomatic-0.36.jar
├── hisat2-2.1.0/          ← HISAT2 binaries
├── output/                ← created at runtime, one sub-dir per sample
└── script.sh              ← main pipeline driver
```

---

## 2. Prerequisites: Tool Installation

Before running the pipeline install the required tools.

| Tool | Version used | Purpose | Install |
|------|-------------|---------|--------|
| **Trimmomatic** | 0.36 | Read quality trimming | http://www.usadellab.org/cms/?page=trimmomatic |
| **HISAT2** | 2.1.0 | Read alignment | https://ccb.jhu.edu/software/hisat2/ |
| **SAMtools** | ≥ 1.9 | BAM manipulation & pileup | http://samtools.sourceforge.net/ |
| **BCFtools** | ≥ 1.9 | Variant calling from pileup | bundled with SAMtools |
| **ANNOVAR** | latest | Variant annotation | http://annovar.openbioinformatics.org/ |

Quick conda install:

```bash
conda create -n snp-pipeline python=3.10
conda activate snp-pipeline
conda install -c bioconda trimmomatic hisat2 samtools bcftools
# ANNOVAR requires manual registration at http://annovar.openbioinformatics.org/
```

---

## 3. Step 1 — Read Trimming with Trimmomatic

Raw sequencing reads often have:
- **Low-quality bases** at the 3' end (quality decreases along the read)
- **Adapter contamination** from library preparation
- **Very short reads** that align unreliably

Trimmomatic removes these issues before alignment.

### 3.1 Pipeline Command

```bash
java -jar $home/Trimmomatic-0.36/trimmomatic-0.36.jar \
    SE \
    -phred33 \
    input.fastq \
    chr_outfile.fastq \
    TRAILING:20 \
    MINLEN:50
```

### 3.2 Parameter Breakdown

| Parameter | Value | Meaning |
|-----------|-------|--------|
| `SE` | — | Single-End mode (one file per sample) |
| `-phred33` | — | Illumina 1.8+ quality encoding (ASCII offset 33) |
| `TRAILING:20` | 20 | Remove trailing bases with Phred quality < 20 |
| `MINLEN:50` | 50 bp | Discard reads shorter than 50 bp after trimming |

**Why TRAILING rather than SLIDINGWINDOW?**  
Quality degradation in Illumina reads is worst at the 3' end. `TRAILING` is fast, simple, and effective for removing these low-quality tails. `SLIDINGWINDOW` is more thorough but slower.

**Phred quality score recap:**

| Phred Q | Error probability | Accuracy |
|---------|-----------------|----------|
| 10 | 1 in 10 | 90% |
| 20 | 1 in 100 | 99% |
| 30 | 1 in 1,000 | 99.9% |
| 40 | 1 in 10,000 | 99.99% |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import random
import re

random.seed(42)
np.random.seed(42)


def phred_to_prob(q):
    """Convert Phred score to error probability."""
    return 10 ** (-q / 10)


def simulate_read_qualities(length=150, seed=42):
    """Simulate a typical Illumina read quality profile."""
    rng = np.random.default_rng(seed)
    # Good quality at start, gradual decline at 3' end
    base_quality = np.linspace(38, 15, length)
    noise = rng.normal(0, 3, length)
    qualities = np.clip(base_quality + noise, 2, 40).astype(int)
    return qualities


def trimmomatic_trailing(qualities, min_q=20):
    """Simulate Trimmomatic TRAILING: remove 3' bases below min_q."""
    trimmed_len = len(qualities)
    for i in range(len(qualities) - 1, -1, -1):
        if qualities[i] >= min_q:
            trimmed_len = i + 1
            break
    return trimmed_len


# Simulate 500 reads and show effect of trimming
read_lengths_before = []
read_lengths_after = []
kept = 0
discarded = 0
MINLEN = 50

for i in range(500):
    q = simulate_read_qualities(150, seed=i)
    read_lengths_before.append(150)
    trimmed = trimmomatic_trailing(q, min_q=20)
    if trimmed >= MINLEN:
        read_lengths_after.append(trimmed)
        kept += 1
    else:
        discarded += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Quality profile of one read
q_example = simulate_read_qualities(150, seed=0)
trim_point = trimmomatic_trailing(q_example, min_q=20)
ax = axes[0]
ax.plot(range(1, 151), q_example, color='steelblue', lw=1.2, label='Quality score')
ax.axhline(y=20, color='red', linestyle='--', lw=1.5, label='TRAILING threshold (Q20)')
ax.axvline(x=trim_point, color='orange', linestyle='-', lw=2, label=f'Trim point ({trim_point} bp)')
ax.fill_betweenx([0, 40], trim_point, 150, alpha=0.15, color='red', label='Removed')
ax.set_xlabel('Position in read (bp)')
ax.set_ylabel('Phred quality score')
ax.set_title('Quality Profile: TRAILING:20 Trimming')
ax.legend(fontsize=8)
ax.set_ylim(0, 42)

# Distribution of read lengths after trimming
ax2 = axes[1]
ax2.hist(read_lengths_after, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=MINLEN, color='red', linestyle='--', lw=2, label=f'MINLEN={MINLEN}')
ax2.set_xlabel('Read length after trimming (bp)')
ax2.set_ylabel('Number of reads')
ax2.set_title(f'Read Length Distribution After Trimming\n(kept={kept}, discarded={discarded})')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Reads kept: {kept} ({100*kept/500:.1f}%)")
print(f"Reads discarded (< {MINLEN} bp): {discarded} ({100*discarded/500:.1f}%)")
print(f"Mean length after trimming: {np.mean(read_lengths_after):.1f} bp")

---

## 4. Step 2 — Alignment with HISAT2

HISAT2 is a fast, splice-aware aligner. For **genomic DNA** (WGS/WES) we disable its RNA-seq-specific features to get clean, ungapped alignments suitable for SNP calling.

### 4.1 Building the Reference Index (builder.sh)

Before the first run, build a HISAT2 index for each reference chromosome:

```bash
#!/bin/bash
# builder.sh — run once per reference
cd Human
for fasta in *.fasta; do
    prefix="${fasta%.fasta}"
    echo "Building index for ${fasta}..."
    hisat2-build "${fasta}" "${prefix}"
done
echo "All indexes built."
```

This produces `.ht2` index files used by the main alignment step.

### 4.2 Alignment Command

```bash
hisat2 \
    -x $home/Human/${sample} \
    -U chr_outfile.fastq \
    --no-spliced-alignment \
    --no-softclip \
    > alignment.sam
```

### 4.3 Key Flags Explained

| Flag | Purpose | Why it matters for SNP calling |
|------|---------|-------------------------------|
| `-x` | Index prefix | Points to the per-chromosome HISAT2 index |
| `-U` | Unpaired (single-end) reads | Matches our SE Trimmomatic output |
| `--no-spliced-alignment` | Disables exon-junction spanning | DNA reads do not span introns; disabling this prevents false split-read alignments |
| `--no-softclip` | Disables soft-clipping | Forces hard alignment; prevents unaligned bases near read ends from masking true variants |

> **Tip:** HISAT2 was designed for RNA-seq. When used on genomic DNA, `--no-spliced-alignment --no-softclip` make it behave more like BWA-MEM or Bowtie2 in end-to-end mode.

In [ ]:
def parse_hisat2_summary(summary_text):
    """Parse HISAT2 alignment summary statistics from stderr output."""
    stats = {}
    lines = summary_text.strip().split('\n')
    for line in lines:
        line = line.strip()
        m = re.match(r'(\d+) reads; of these:', line)
        if m:
            stats['total_reads'] = int(m.group(1))
        m = re.match(r'(\d+) \((.*?)%\) aligned exactly 1 time', line)
        if m:
            stats['uniquely_aligned'] = int(m.group(1))
            stats['unique_pct'] = float(m.group(2))
        m = re.match(r'(\d+) \((.*?)%\) aligned >1 times', line)
        if m:
            stats['multi_aligned'] = int(m.group(1))
            stats['multi_pct'] = float(m.group(2))
        m = re.match(r'(.*?)% overall alignment rate', line)
        if m:
            stats['overall_rate'] = float(m.group(1))
    return stats


# Typical HISAT2 summary from a WGS sample
hisat2_output = """
2500000 reads; of these:
  2500000 (100.00%) were unpaired; of these:
    62500 (2.50%) aligned 0 times
    2225000 (89.00%) aligned exactly 1 time
    212500 (8.50%) aligned >1 times
97.50% overall alignment rate
"""

stats = parse_hisat2_summary(hisat2_output)
total = stats.get('total_reads', 0)
unique = stats.get('uniquely_aligned', 0)
multi = stats.get('multi_aligned', 0)
unaligned = total - unique - multi

print("HISAT2 Alignment Summary")
print("=" * 40)
print(f"Total reads:           {total:>10,}")
print(f"Uniquely aligned:      {unique:>10,}  ({100*unique/total:.1f}%)")
print(f"Multi-mapped:          {multi:>10,}  ({100*multi/total:.1f}%)")
print(f"Unaligned:             {unaligned:>10,}  ({100*unaligned/total:.1f}%)")
print(f"Overall rate:          {stats.get('overall_rate', 0):>9.1f}%")
print()

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
categories = ['Uniquely\naligned', 'Multi-\nmapped', 'Unaligned']
counts = [unique, multi, unaligned]
colors = ['#2196F3', '#FF9800', '#F44336']
bars = ax.bar(categories, [c/total*100 for c in counts], color=colors, edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{count/total*100:.1f}%\n({count:,})', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Percentage of reads (%)')
ax.set_title('HISAT2 Alignment Rate')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

# Quality thresholds
print("Alignment quality assessment:")
if stats.get('overall_rate', 0) >= 90:
    print("  ✓ Overall rate >= 90%: GOOD")
elif stats.get('overall_rate', 0) >= 70:
    print("  ⚠ Overall rate 70-90%: ACCEPTABLE (check for contamination)")
else:
    print("  ✗ Overall rate < 70%: POOR (wrong reference or severe contamination)")
if unique / total >= 0.80:
    print("  ✓ Unique alignment rate >= 80%: GOOD")
else:
    print("  ⚠ High multi-mapping rate: consider duplicate-region filtering")

---

## 5. Step 3 — BAM Processing with samtools

The SAM file from HISAT2 must be converted, sorted, and indexed before variant calling.

### 5.1 SAM → BAM → Sort → Index

```bash
# Convert SAM to compressed BAM
samtools view alignment.sam -b -o alignment.bam

# Sort by genomic coordinate (required for mpileup)
samtools sort alignment.bam -T out_sort.txt -o alignment_sorted.bam

# Create index for random access
samtools index alignment_sorted.bam
```

### 5.2 Alignment Statistics

```bash
# Per-chromosome read counts and unmapped reads
samtools idxstats alignment_sorted.bam > out.txt

# Per-base coverage depth
samtools depth alignment_sorted.bam > depth_sorted.tsv
```

**`idxstats` output format:**
```
CHROM  LENGTH  MAPPED  UNMAPPED
chr1   248956422  1250000  0
chr2   242193529  980000   0
...
*      0          0        62500
```

### 5.3 Creating the Pileup with mpileup

```bash
# -u: uncompressed BCF output (piped to bcftools)
# -f: reference FASTA for consensus calling
samtools mpileup -uf $home/Human/${sample}.fasta alignment_sorted.bam > snp.bcf
```

`mpileup` computes a per-base summary of all reads covering each position: the reference base, each read's base call, and its quality score. This pileup is the input to `bcftools call`.

**mpileup column format:**
```
CHROM  POS   REF  DEPTH  BASES  QUALS
chr1   10000  A    30     .,..,^].  IIIIIFFF
```

| Symbol | Meaning |
|--------|--------|
| `.` | Match to forward strand |
| `,` | Match to reverse strand |
| `A/T/G/C` | Mismatch (uppercase = forward, lowercase = reverse) |
| `^` | Start of read (followed by mapping quality) |
| `$` | End of read |
| `+nSeq` | Insertion of n bases |
| `-nSeq` | Deletion of n bases |

In [ ]:
def simulate_idxstats(chroms, total_reads=2_000_000, seed=42):
    """Simulate samtools idxstats output."""
    rng = np.random.default_rng(seed)
    # Chromosome lengths (hg19 main chromosomes)
    lengths = {
        'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430,
        'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067,
        'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431,
        'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895,
        'chrX': 155270560, 'chrY': 59373566,
    }
    # Proportionally distribute reads by chromosome length
    total_len = sum(lengths[c] for c in chroms if c in lengths)
    rows = []
    assigned = 0
    for chrom in chroms:
        length = lengths.get(chrom, 50_000_000)
        expected = int(total_reads * length / total_len)
        noise = int(rng.normal(0, expected * 0.05))
        mapped = max(0, expected + noise)
        assigned += mapped
        rows.append((chrom, length, mapped, 0))
    rows.append(('*', 0, 0, total_reads - assigned))  # unmapped
    return rows


def simulate_depth_profile(length=1000, mean_depth=30, seed=42):
    """Simulate samtools depth output for a genomic region."""
    rng = np.random.default_rng(seed)
    depths = rng.negative_binomial(n=mean_depth, p=0.5, size=length)
    return depths


chroms = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5',
          'chr6', 'chr7', 'chr8', 'chr9', 'chr10',
          'chr11', 'chr12', 'chrX', 'chrY']

idx = simulate_idxstats(chroms)

print(f"{'CHROM':<8} {'LENGTH':>12} {'MAPPED':>10} {'UNMAPPED':>10}")
print("-" * 46)
for row in idx:
    print(f"{row[0]:<8} {row[1]:>12,} {row[2]:>10,} {row[3]:>10,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-chromosome read counts
ax = axes[0]
mapped_chroms = [(r[0], r[2]) for r in idx if r[0] != '*']
names, counts = zip(*mapped_chroms)
ax.bar(range(len(names)), counts, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mapped reads')
ax.set_title('Reads per Chromosome (idxstats)')

# Coverage depth profile
ax2 = axes[1]
depths = simulate_depth_profile(length=500, mean_depth=30)
ax2.fill_between(range(len(depths)), depths, alpha=0.6, color='steelblue')
ax2.axhline(y=np.mean(depths), color='red', linestyle='--',
            label=f'Mean depth = {np.mean(depths):.1f}x')
ax2.axhline(y=10, color='orange', linestyle=':', label='Min. depth threshold (10x)')
ax2.set_xlabel('Genomic position')
ax2.set_ylabel('Read depth')
ax2.set_title('Coverage Depth Profile (samtools depth)')
ax2.legend()

plt.tight_layout()
plt.show()

total_mapped = sum(r[2] for r in idx if r[0] != '*')
print(f"\nTotal mapped reads: {total_mapped:,}")
print(f"Mean coverage depth: {np.mean(depths):.1f}x")
print(f"Bases >= 10x coverage: {np.sum(depths >= 10) / len(depths) * 100:.1f}%")

---

## 6. Step 4 — Variant Calling with bcftools

### 6.1 Calling Variants

```bash
bcftools call -cv snp.bcf -o snp.vcf
```

| Flag | Meaning |
|------|--------|
| `-c` | Consensus caller (original samtools model; simpler than the `-m` multiallelic caller) |
| `-v` | Output **variant sites only** (skip reference-matching positions) |

### 6.2 Consensus Caller (`-c`) vs Multiallelic Caller (`-m`)

| Feature | `-c` consensus | `-m` multiallelic |
|---------|---------------|------------------|
| Alleles per site | 1 ALT | Multiple ALTs |
| Speed | Faster | Slower |
| Best for | Simple diploid SNP surveys | Cohorts, complex variants |
| Recommended for | Quick monogenic/WGAS | GATK Best Practices pipelines |

This pipeline uses `-c` for speed, making it suitable for monogenic disease screening and whole-genome association surveys.

### 6.3 Example VCF Output

```
##fileformat=VCFv4.1
##FILTER=<ID=PASS,Description="All filters passed">
##bcftoolsVersion=1.15
##INFO=<ID=DP,Number=1,Type=Integer,Description="Raw read depth">
##INFO=<ID=DP4,Number=4,Type=Integer,Description="Number of high-quality ref-fwd, ref-rev, alt-fwd, alt-rev bases">
##INFO=<ID=MQ,Number=1,Type=Integer,Description="Root-mean-square mapping quality of covering reads">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="List of Phred-scaled genotype likelihoods">
#CHROM  POS     ID  REF  ALT  QUAL  FILTER  INFO
chr1    925952  .   G    A    222   .       DP=30;DP4=0,0,15,15;MQ=60
chr1    931279  .   T    C    199   .       DP=25;DP4=0,0,12,13;MQ=58
```

The `DP4` field is key: `ref-fwd, ref-rev, alt-fwd, alt-rev`. Balanced counts across both strands indicate a real variant; strong strand bias suggests a sequencing artifact.

In [ ]:
def generate_simulated_vcf(n_variants=20, seed=42):
    """Generate a simulated bcftools VCF output."""
    rng = np.random.default_rng(seed)
    bases = ['A', 'T', 'G', 'C']
    chromosomes = ['chr1', 'chr2', 'chr3', 'chr7', 'chr17']

    variants = []
    for i in range(n_variants):
        chrom = rng.choice(chromosomes)
        pos = int(rng.integers(100_000, 50_000_000))
        ref = rng.choice(bases)
        alt_choices = [b for b in bases if b != ref]
        alt = rng.choice(alt_choices)
        qual = int(rng.integers(50, 999))
        depth = int(rng.integers(10, 80))
        # DP4: ref-fwd, ref-rev, alt-fwd, alt-rev
        # Simulate mostly alt reads for a called variant
        alt_total = int(depth * rng.uniform(0.4, 1.0))
        ref_total = depth - alt_total
        rf = int(ref_total * rng.uniform(0.3, 0.7))
        rr = ref_total - rf
        af = int(alt_total * rng.uniform(0.3, 0.7))
        ar = alt_total - af
        mq = int(rng.integers(40, 60))
        variants.append({
            'chrom': chrom, 'pos': pos, 'ref': ref, 'alt': alt,
            'qual': qual, 'depth': depth,
            'dp4': (rf, rr, af, ar), 'mq': mq
        })
    return sorted(variants, key=lambda v: (v['chrom'], v['pos']))


def strand_bias_ratio(dp4):
    """Compute alt strand bias ratio. Values far from 0.5 indicate bias."""
    rf, rr, af, ar = dp4
    alt_total = af + ar
    if alt_total == 0:
        return None
    return af / alt_total


variants = generate_simulated_vcf(n_variants=20)

print(f"{'CHROM':<7} {'POS':>10} {'REF'} {'ALT'} {'QUAL':>5} {'DP':>4} {'DP4':>22} {'StrandBias':>11}")
print("-" * 72)
for v in variants:
    dp4_str = ','.join(map(str, v['dp4']))
    sb = strand_bias_ratio(v['dp4'])
    sb_flag = '⚠ BIAS' if sb is not None and (sb < 0.2 or sb > 0.8) else 'OK'
    print(f"{v['chrom']:<7} {v['pos']:>10} {v['ref']}   {v['alt']}  "
          f"{v['qual']:>5}  {v['depth']:>3} {dp4_str:>22} {sb_flag:>11}")

# Visualise quality distribution
quals = [v['qual'] for v in variants]
depths = [v['depth'] for v in variants]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(quals, bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(x=30, color='red', linestyle='--', label='Q30 threshold')
axes[0].set_xlabel('Variant QUAL score')
axes[0].set_ylabel('Count')
axes[0].set_title('Variant Quality Distribution')
axes[0].legend()

axes[1].scatter(depths, quals, alpha=0.7, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Read depth (DP)')
axes[1].set_ylabel('QUAL score')
axes[1].set_title('QUAL vs Depth')

plt.tight_layout()
plt.show()

---

## 7. Step 5 — Variant Annotation with ANNOVAR

ANNOVAR (ANNOtate VARiation) is a widely-used Perl-based tool for annotating genetic variants against multiple databases. Unlike VEP or SnpEff, ANNOVAR is especially common in clinical genetics workflows.

### 7.1 Converting VCF to ANNOVAR Input

```bash
perl annovar/convert2annovar.pl \
    -format vcf4 \
    snp.vcf > snp.avinput
```

The `.avinput` format is a tab-delimited file with columns:
```
chr  start  end  ref  alt  [extra VCF fields...]
chr1  925952  925952  G  A  0  het  30
```

### 7.2 Annotation Steps

The pipeline runs five ANNOVAR annotation passes:

```bash
# 1. Flag against dbSNP138 (known SNPs)
perl annovar/annotate_variation.pl \
    -filter -out snp.rs.SAMPLE -build hg19 \
    -dbtype snp138 snp.avinput annovar/humandb/

# 2. RefGene annotation (gene / exon / UTR / intron / intergenic)
perl annovar/annotate_variation.pl \
    -out snp.refgene.SAMPLE -build hg19 \
    snp.avinput annovar/humandb/

# 3. 1000 Genomes allele frequencies
perl annovar/annotate_variation.pl \
    -filter -out snp.1000g.SAMPLE -buildver hg19 \
    -dbtype 1000g2014oct_all snp.avinput annovar/humandb/

# 4. GWAS Catalog (trait associations)
perl annovar/annotate_variation.pl \
    -regionanno -out snp.gwas.SAMPLE -build hg19 \
    -dbtype gwasCatalog snp.avinput annovar/humandb/

# 5. ClinVar (clinical significance)
perl annovar/annotate_variation.pl \
    -filter -out snp.clin.SAMPLE -dbtype clinvar_20150629 \
    -buildver hg19 snp.avinput annovar/humandb/
```

### 7.3 ANNOVAR Annotation Modes

| Mode | Flag | Purpose | Output |
|------|------|---------|-------|
| **gene-based** | (default) | Assign variant to gene region | `exonic`, `intronic`, `UTR3`, `UTR5`, `intergenic`, `splicing` |
| **filter-based** | `-filter` | Check if variant is in a database | Two files: `_dropped` (found) and `_filtered` (not found) |
| **region-based** | `-regionanno` | Check if variant overlaps a genomic region | Overlapping entries from the database |

### 7.4 Downloading ANNOVAR Databases

```bash
# Download all databases used in this pipeline (hg19)
perl annotate_variation.pl -buildver hg19 -downdb -webfrom annovar snp138 humandb/
perl annotate_variation.pl -buildver hg19 -downdb refGene humandb/
perl annotate_variation.pl -buildver hg19 -downdb -webfrom annovar 1000g2014oct humandb/
perl annotate_variation.pl -buildver hg19 -downdb -webfrom annovar gwasCatalog humandb/
perl annotate_variation.pl -buildver hg19 -downdb -webfrom annovar clinvar_20150629 humandb/
```

In [ ]:
# Simulate ANNOVAR avinput format
def vcf_to_avinput(variants):
    """
    Convert VCF-like variant list to ANNOVAR .avinput format.
    Format: chr  start  end  ref  alt  quality  zygosity  depth
    """
    rows = []
    for v in variants:
        # For SNPs: start == end == pos
        rows.append({
            'chr': v['chrom'],
            'start': v['pos'],
            'end': v['pos'],
            'ref': v['ref'],
            'alt': v['alt'],
            'qual': v['qual'],
            'depth': v['depth'],
        })
    return rows


avinput = vcf_to_avinput(variants)
print("Sample .avinput file (ANNOVAR input):")
print(f"{'chr':<7} {'start':>10} {'end':>10} {'ref'} {'alt'} {'qual':>5} {'dp':>4}")
print("-" * 48)
for row in avinput[:8]:
    print(f"{row['chr']:<7} {row['start']:>10} {row['end']:>10} "
          f"{row['ref']}   {row['alt']}  {row['qual']:>5}  {row['depth']:>3}")
print(f"... ({len(avinput)} total variants)")

---

## 8. Step 5 (continued) — Understanding ANNOVAR Output

### 8.1 RefGene Annotation Output

The RefGene pass produces a `.variant_function` file:

```
intergenic  BRCA1,BRCA2  chr17  41196311  41196311  G  A  ...
exonic      TP53          chr17  7578406   7578406   C  T  ...
intronic    EGFR          chr7   55191822  55191822  A  G  ...
UTR3        KRAS          chr12  25398284  25398284  G  T  ...
```

And an `.exonic_variant_function` file for coding variants:

```
line2  nonsynonymous SNV  TP53:NM_000546:exon7:c.C817T:p.R273C  chr17 ...
line5  synonymous SNV     EGFR:NM_005228:exon5:c.T2145C:p.I715I  chr7 ...
line8  stopgain           APC:NM_000038:exon15:c.C3927A:p.Y1309*  chr5 ...
```

### 8.2 Filter-based Output (dbSNP, 1000G, ClinVar)

Filter-based passes produce two files:

| File | Content |
|------|---------|
| `_dropped` | Variants **found** in the database |
| `_filtered` | Variants **not found** (potentially novel) |

Example `_dropped` line (1000G):
```
1000g2014oct_all  0.0462  chr1  925952  925952  G  A  ...
```
The second column is the **allele frequency** in the 1000 Genomes dataset.

### 8.3 GWAS Catalog Output

The `-regionanno` pass produces a file where each variant overlapping a GWAS hit is annotated:

```
Name=rs7903146;Trait=Type_2_diabetes;P-value=5e-15;OR=1.40
Name=rs9939609;Trait=Obesity,BMI;P-value=2e-20;OR=1.31
```

This file is aggregated across all samples into `log_SNP.csv`, giving a quick overview of GWAS-relevant variants.

In [ ]:
# Simulate ANNOVAR annotation results and parse them
GENE_REGIONS = ['exonic', 'intronic', 'UTR3', 'UTR5', 'intergenic', 'splicing', 'ncRNA_intronic']
EXONIC_TYPES = ['nonsynonymous SNV', 'synonymous SNV', 'stopgain', 'stoploss', 'frameshift deletion',
                'frameshift insertion', 'nonframeshift deletion']
GENES = ['TP53', 'BRCA1', 'BRCA2', 'EGFR', 'KRAS', 'APC', 'MLH1', 'MSH2', 'ATM', 'PTEN',
         'BRAF', 'PIK3CA', 'CDKN2A', 'RB1', 'VHL', None]


def simulate_annovar_results(variants, seed=42):
    """Simulate realistic ANNOVAR annotation results."""
    rng = np.random.default_rng(seed)
    # Region probabilities (realistic for WGS)
    region_probs = [0.03, 0.40, 0.08, 0.02, 0.42, 0.02, 0.03]

    annotated = []
    for v in variants:
        region = rng.choice(GENE_REGIONS, p=region_probs)
        gene = rng.choice(GENES)
        gene = gene if gene else 'INTERGENIC'

        exonic_type = None
        if region == 'exonic':
            exon_probs = [0.45, 0.35, 0.05, 0.02, 0.05, 0.05, 0.03]
            exonic_type = rng.choice(EXONIC_TYPES, p=exon_probs)

        # 1000G allele frequency
        af_1000g = None
        if rng.random() < 0.6:  # 60% of variants are in 1000G
            af_1000g = float(rng.beta(0.5, 10))  # Most common variants are rare

        # ClinVar significance
        clinsig = None
        if rng.random() < 0.15:
            clinsig = rng.choice(['Benign', 'Likely_benign', 'Uncertain_significance',
                                  'Likely_pathogenic', 'Pathogenic'],
                                 p=[0.30, 0.25, 0.30, 0.10, 0.05])

        # GWAS hit
        gwas_trait = None
        if rng.random() < 0.10:
            gwas_trait = rng.choice(['Type_2_diabetes', 'Coronary_artery_disease',
                                     'BMI', 'Hypertension', 'Alzheimers_disease'])

        # dbSNP rsID
        rs_id = f'rs{rng.integers(100000, 9999999)}' if rng.random() < 0.65 else None

        annotated.append({
            **v,
            'region': region,
            'gene': gene,
            'exonic_type': exonic_type,
            'af_1000g': af_1000g,
            'clinsig': clinsig,
            'gwas_trait': gwas_trait,
            'rs_id': rs_id,
        })
    return annotated


annotated = simulate_annovar_results(variants)

# Summary statistics
region_counts = defaultdict(int)
for v in annotated:
    region_counts[v['region']] += 1

exonic_type_counts = defaultdict(int)
for v in annotated:
    if v['exonic_type']:
        exonic_type_counts[v['exonic_type']] += 1

clinvar_counts = defaultdict(int)
for v in annotated:
    if v['clinsig']:
        clinvar_counts[v['clinsig']] += 1

print("=== ANNOVAR Annotation Summary ===")
print(f"\nTotal variants: {len(annotated)}")
print(f"In dbSNP138:   {sum(1 for v in annotated if v['rs_id'])} ({100*sum(1 for v in annotated if v['rs_id'])/len(annotated):.0f}%)")
print(f"In 1000G:      {sum(1 for v in annotated if v['af_1000g'] is not None)} ({100*sum(1 for v in annotated if v['af_1000g'] is not None)/len(annotated):.0f}%)")
print(f"In ClinVar:    {sum(1 for v in annotated if v['clinsig'])} ({100*sum(1 for v in annotated if v['clinsig'])/len(annotated):.0f}%)")
print(f"GWAS hits:     {sum(1 for v in annotated if v['gwas_trait'])} ({100*sum(1 for v in annotated if v['gwas_trait'])/len(annotated):.0f}%)")

print("\nRegion distribution:")
for region, cnt in sorted(region_counts.items(), key=lambda x: -x[1]):
    print(f"  {region:<25} {cnt:>3}")

if exonic_type_counts:
    print("\nExonic variant types:")
    for etype, cnt in sorted(exonic_type_counts.items(), key=lambda x: -x[1]):
        print(f"  {etype:<30} {cnt:>3}")

In [ ]:
# Visualise ANNOVAR annotation results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Region distribution pie
ax = axes[0, 0]
regions = list(region_counts.keys())
counts = [region_counts[r] for r in regions]
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(regions)))
wedges, texts, autotexts = ax.pie(
    counts, labels=regions, colors=colors_pie, autopct='%1.0f%%',
    startangle=90, textprops={'fontsize': 8}
)
ax.set_title('Variant Location (RefGene)')

# 2. ClinVar significance
ax2 = axes[0, 1]
if clinvar_counts:
    sig_order = ['Pathogenic', 'Likely_pathogenic', 'Uncertain_significance',
                 'Likely_benign', 'Benign']
    sig_colors = ['#D32F2F', '#FF7043', '#FFC107', '#66BB6A', '#2196F3']
    present = [(s, clinvar_counts[s], c) for s, c in zip(sig_order, sig_colors) if s in clinvar_counts]
    if present:
        labels, vals, cols = zip(*present)
        ax2.barh(range(len(labels)), vals, color=cols, edgecolor='white')
        ax2.set_yticks(range(len(labels)))
        ax2.set_yticklabels([l.replace('_', ' ') for l in labels])
        ax2.set_xlabel('Number of variants')
        ax2.set_title('ClinVar Clinical Significance')
else:
    ax2.text(0.5, 0.5, 'No ClinVar variants', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('ClinVar Clinical Significance')

# 3. 1000G allele frequency distribution
ax3 = axes[1, 0]
af_values = [v['af_1000g'] for v in annotated if v['af_1000g'] is not None]
if af_values:
    ax3.hist(af_values, bins=20, color='steelblue', edgecolor='white', log=True)
    ax3.set_xlabel('1000 Genomes allele frequency')
    ax3.set_ylabel('Count (log scale)')
    ax3.set_title('1000G Allele Frequency Distribution')
    ax3.axvline(x=0.01, color='red', linestyle='--', label='1% threshold')
    ax3.legend()

# 4. GWAS traits
ax4 = axes[1, 1]
gwas_variants = [v for v in annotated if v['gwas_trait']]
if gwas_variants:
    trait_counts = defaultdict(int)
    for v in gwas_variants:
        trait_counts[v['gwas_trait']] += 1
    traits = list(trait_counts.keys())
    tcounts = [trait_counts[t] for t in traits]
    ax4.barh(range(len(traits)), tcounts, color='coral', edgecolor='white')
    ax4.set_yticks(range(len(traits)))
    ax4.set_yticklabels([t.replace('_', ' ') for t in traits])
    ax4.set_xlabel('Number of variants')
    ax4.set_title('GWAS Catalog Trait Associations')
else:
    ax4.text(0.5, 0.5, 'No GWAS hits', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('GWAS Catalog Trait Associations')

plt.suptitle('ANNOVAR Annotation Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## 9. Step 6 — Aggregating Results: The Full Pipeline Report

After processing all samples, the pipeline aggregates GWAS hits across samples:

```bash
cd ./output
for sample_dir in */; do
    cat ./${sample_dir}/snp.gwas.${sample_dir%.}/hg19_gwasCatalog >> log_SNP.csv
done
```

### 9.1 Interpreting the Final Report

A clinically-oriented SNP report should prioritise variants by:

1. **ClinVar Pathogenic / Likely Pathogenic** — immediate clinical relevance
2. **Novel variants** (not in dbSNP / 1000G) in coding exons — potential disease-causing
3. **Rare variants** (1000G AF < 1%) with predicted functional impact
4. **GWAS hits** — phenotype association
5. **Common variants** (1000G AF > 5%) — typically benign background variation

In [ ]:
def prioritise_variants(annotated):
    """
    Prioritise annotated variants for clinical review.
    Returns variants sorted by priority tier (Tier 1 = highest).
    """
    results = []
    for v in annotated:
        tier = 5  # default: common / intergenic
        reason = ''

        clinsig = v.get('clinsig')
        exonic_type = v.get('exonic_type')
        af = v.get('af_1000g')
        rs_id = v.get('rs_id')
        gwas = v.get('gwas_trait')
        region = v.get('region')

        if clinsig in ('Pathogenic', 'Likely_pathogenic'):
            tier = 1
            reason = f'ClinVar:{clinsig}'
        elif (
            region == 'exonic'
            and exonic_type in ('stopgain', 'stoploss', 'frameshift deletion', 'frameshift insertion')
            and rs_id is None
        ):
            tier = 1
            reason = f'Novel:{exonic_type}'
        elif (
            region == 'exonic'
            and exonic_type == 'nonsynonymous SNV'
            and (af is None or af < 0.01)
        ):
            tier = 2
            reason = 'Rare nonsynonymous SNV'
        elif clinsig == 'Uncertain_significance' and region == 'exonic':
            tier = 2
            reason = 'ClinVar:VUS exonic'
        elif gwas:
            tier = 3
            reason = f'GWAS:{gwas}'
        elif region in ('splicing', 'exonic') and (af is None or af < 0.05):
            tier = 3
            reason = f'Rare {region} variant'
        elif region in ('UTR3', 'UTR5'):
            tier = 4
            reason = f'{region} variant'
        elif af is not None and af >= 0.05:
            tier = 5
            reason = f'Common (AF={af:.3f})'
        else:
            tier = 4
            reason = f'{region}'

        results.append({**v, 'tier': tier, 'reason': reason})

    return sorted(results, key=lambda x: (x['tier'], -x['qual']))


prioritised = prioritise_variants(annotated)

tier_colors = {1: '🔴', 2: '🟠', 3: '🟡', 4: '🔵', 5: '⚪'}
tier_labels = {
    1: 'Tier 1 — Likely pathogenic / high impact',
    2: 'Tier 2 — Possibly pathogenic',
    3: 'Tier 3 — GWAS / rare coding',
    4: 'Tier 4 — UTR / background',
    5: 'Tier 5 — Common / intergenic',
}

tier_counts = defaultdict(int)
for v in prioritised:
    tier_counts[v['tier']] += 1

print("=== Variant Prioritisation Report ===")
for tier in sorted(tier_counts):
    print(f"  {tier_colors[tier]} {tier_labels[tier]}: {tier_counts[tier]} variant(s)")

print("\n--- Top Priority Variants ---")
print(f"{'Tier':<5} {'CHROM':<7} {'POS':>10} {'Gene':<10} {'Region':<12} {'Exonic Type':<22} {'Reason':<30} {'RS_ID'}")
print("-" * 110)
for v in prioritised[:10]:
    print(f"{tier_colors[v['tier']]} T{v['tier']}  {v['chrom']:<7} {v['pos']:>10} "
          f"{v['gene']:<10} {v['region']:<12} {(v['exonic_type'] or '-'):<22} "
          f"{v['reason']:<30} {v['rs_id'] or 'novel'}")

---

## 10. The Complete Pipeline Script

Here is the full `script.sh` from the pipeline, with annotations explaining each command:

```bash
#!/bin/bash
echo START
home=`pwd`              # Store working directory
cd reads                # Enter reads directory
a=*.fas*                # Match all .fastq / .fasta files
export PATH=${PATH}:$home/hisat2-2.1.0   # Add HISAT2 to PATH

for x in $a; do

    # --- Setup ---
    mkdir -p output/${x%.fa*}            # Create per-sample output directory
    cp $x ./output/${x%.fa*}            # Copy reads to output dir
    cd ./output/${x%.fa*}

    # --- Step 1: Trimming ---
    java -jar $home/Trimmomatic-0.36/trimmomatic-0.36.jar \
        SE -phred33 $x chr_outfile.fastq TRAILING:20 MINLEN:50

    # --- Step 2: Alignment ---
    hisat2 -x $home/Human/${x%.fa*} -U chr_outfile.fastq \
        --no-spliced-alignment --no-softclip > alignment.sam

    # --- Step 3: BAM Processing ---
    samtools view alignment.sam -b -o alignment.bam          # SAM → BAM
    samtools sort alignment.bam -T out_sort.txt \
        -o alignment_sorted.bam                              # Sort
    samtools index alignment_sorted.bam                      # Index
    samtools idxstats alignment_sorted.bam > out.txt         # Chrom stats
    samtools mpileup -uf $home/Human/${x%.fa*}.fasta \
        alignment_sorted.bam > snp.bcf                       # Pileup
    samtools depth alignment_sorted.bam > depth_sorted.tsv   # Coverage

    # --- Step 4: Variant Calling ---
    bcftools call -cv snp.bcf -o snp.vcf

    # --- Step 5: ANNOVAR Annotation ---
    # 5a. Convert VCF to ANNOVAR input format
    perl `pwd`/annovar/convert2annovar.pl -format vcf4 snp.vcf > snp.avinput

    # 5b. dbSNP138 — flag known SNPs
    perl `pwd`/annovar/annotate_variation.pl \
        -filter -out snp.rs.${x%.fa*} -build hg19 -dbtype snp138 \
        snp.avinput $home/annovar/humandb/

    # 5c. RefGene — gene-based annotation
    perl /nfs/srv/databases/annovar/annotate_variation.pl \
        -out snp.refgene.${x%.fa*} -build hg19 \
        snp.avinput $home/annovar/humandb/

    # 5d. 1000G — population frequency
    perl /nfs/srv/databases/annovar/annotate_variation.pl \
        -filter -out snp.1000g.${x%.fa*} -buildver hg19 -dbtype 1000g2014oct_all \
        snp.avinput $home/annovar/humandb/

    # 5e. GWAS Catalog — trait associations
    perl /nfs/srv/databases/annovar/annotate_variation.pl \
        -regionanno -out snp.gwas.${x%.fa*} -build hg19 -dbtype gwasCatalog \
        snp.avinput $home/annovar/humandb/

    # 5f. ClinVar — clinical significance
    perl /nfs/srv/databases/annovar/annotate_variation.pl \
        -filter -out snp.clin.${x%.fa*} -dbtype clinvar_20150629 -buildver hg19 \
        snp.avinput $home/annovar/humandb/

    cd ../..
    echo Complete for sample: $x
done

# --- Step 6: Aggregate GWAS results across all samples ---
cd ./output
for x in $(ls); do
    cat ./$x/snp.gwas.$x.hg19_gwasCatalog >> log_SNP.csv
done

echo END
```

---

## 11. Practical Exercises

### Exercise 1: Trimming Parameter Effects
Modify the `trimmomatic_trailing` function to implement `SLIDINGWINDOW:4:20` trimming (window size 4, minimum average quality 20) and compare the number of retained reads to the `TRAILING:20` approach.

### Exercise 2: Strand Bias Filtering
Write a function that filters variants from the simulated VCF based on strand bias: keep only variants where the alt-forward ratio is between 0.2 and 0.8. How many variants survive?

### Exercise 3: Rare Variant Enrichment
From the annotated variant list:
- Extract all exonic variants with 1000G AF < 0.01 (or absent from 1000G)
- Separate synonymous from nonsynonymous
- Calculate the ratio of nonsynonymous to synonymous (dN/dS proxy)

### Exercise 4: Multi-sample Aggregation
Simulate three patients each with 50 variants and:
- Find variants shared by all three (potential germline)
- Find variants unique to one patient (potential de novo or somatic)
- Annotate the shared variants with the pipeline's outputs

In [ ]:
# Exercise 1 starter: SLIDINGWINDOW trimming
def trimmomatic_slidingwindow(qualities, window_size=4, min_avg_q=20):
    """
    Simulate Trimmomatic SLIDINGWINDOW:window_size:min_avg_q.
    Scan from 5' to 3'; cut when the window average drops below min_avg_q.
    """
    for i in range(len(qualities) - window_size + 1):
        window = qualities[i:i + window_size]
        if np.mean(window) < min_avg_q:
            return i  # trim from position i onwards
    return len(qualities)  # no trimming needed


# Compare TRAILING:20 vs SLIDINGWINDOW:4:20
trailing_kept, trailing_lengths = 0, []
sliding_kept, sliding_lengths = 0, []

for i in range(500):
    q = simulate_read_qualities(150, seed=i)

    t_len = trimmomatic_trailing(q, min_q=20)
    if t_len >= 50:
        trailing_kept += 1
        trailing_lengths.append(t_len)

    s_len = trimmomatic_slidingwindow(q, window_size=4, min_avg_q=20)
    if s_len >= 50:
        sliding_kept += 1
        sliding_lengths.append(s_len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(trailing_lengths, bins=25, alpha=0.7, color='steelblue', label=f'TRAILING:20 (kept={trailing_kept})')
axes[0].hist(sliding_lengths, bins=25, alpha=0.7, color='coral', label=f'SLIDINGWINDOW:4:20 (kept={sliding_kept})')
axes[0].set_xlabel('Read length after trimming (bp)')
axes[0].set_ylabel('Count')
axes[0].set_title('Trimming Method Comparison')
axes[0].legend()

axes[1].boxplot([trailing_lengths, sliding_lengths], labels=['TRAILING:20', 'SLIDINGWINDOW:4:20'])
axes[1].set_ylabel('Read length after trimming (bp)')
axes[1].set_title('Length Distribution Boxplot')

plt.tight_layout()
plt.show()

print(f"TRAILING:20       - kept {trailing_kept}/500, mean length {np.mean(trailing_lengths):.1f} bp")
print(f"SLIDINGWINDOW:4:20 - kept {sliding_kept}/500, mean length {np.mean(sliding_lengths):.1f} bp")
print("\nNote: SLIDINGWINDOW is more aggressive — it trims as soon as quality drops,")
print("not just at the very end. This usually produces shorter but cleaner reads.")

---

## 12. Summary

### Pipeline Recap

| Step | Tool | Input | Output | Key options |
|------|------|-------|--------|------------|
| 1. QC | Trimmomatic | `.fastq` | trimmed `.fastq` | `SE TRAILING:20 MINLEN:50` |
| 2. Align | HISAT2 | trimmed `.fastq` | `.sam` | `--no-spliced-alignment --no-softclip` |
| 3a. Convert | samtools view | `.sam` | `.bam` | `-b` |
| 3b. Sort | samtools sort | `.bam` | sorted `.bam` | `-T tmp` |
| 3c. Index | samtools index | sorted `.bam` | `.bam.bai` | |
| 3d. Stats | samtools idxstats / depth | `.bam` | `.txt` / `.tsv` | |
| 3e. Pileup | samtools mpileup | sorted `.bam` + ref | `.bcf` | `-uf` |
| 4. Call | bcftools call | `.bcf` | `.vcf` | `-cv` |
| 5a. Convert | ANNOVAR convert2annovar | `.vcf` | `.avinput` | `-format vcf4` |
| 5b. dbSNP | ANNOVAR annotate_variation | `.avinput` | `_dropped/_filtered` | `-filter -dbtype snp138` |
| 5c. RefGene | ANNOVAR annotate_variation | `.avinput` | `.variant_function` | gene-based |
| 5d. 1000G | ANNOVAR annotate_variation | `.avinput` | `_dropped/_filtered` | `-filter -dbtype 1000g2014oct_all` |
| 5e. GWAS | ANNOVAR annotate_variation | `.avinput` | annotated file | `-regionanno -dbtype gwasCatalog` |
| 5f. ClinVar | ANNOVAR annotate_variation | `.avinput` | `_dropped/_filtered` | `-filter -dbtype clinvar_*` |
| 6. Report | bash | per-sample `.csv` | `log_SNP.csv` | |

### Strengths of This Pipeline
- **Simple and scriptable**: pure shell, easy to extend
- **Multi-database annotation**: covers population frequency, clinical significance, and trait associations in one run
- **Per-sample parallelisable**: the inner loop can be trivially parallelised with GNU parallel

### Limitations and Improvements
- **HISAT2** is optimised for RNA-seq; for WGS/WES, **BWA-MEM2** or **Bowtie2** are preferred
- **bcftools call -c** is a legacy caller; **GATK HaplotypeCaller** or **DeepVariant** are more accurate
- No **duplicate marking** step (samtools markdup / Picard MarkDuplicates)
- No **BQSR** (Base Quality Score Recalibration)
- ANNOVAR databases have specific build versions: always match `hg19` vs `hg38`

### Further Reading

- **Trimmomatic**: Bolger et al. (2014), Bioinformatics. https://doi.org/10.1093/bioinformatics/btu170
- **HISAT2**: Kim et al. (2019), Nature Methods. https://doi.org/10.1038/s41592-019-0521-4
- **SAMtools/BCFtools**: Li et al. (2009), Bioinformatics. https://doi.org/10.1093/bioinformatics/btp352
- **ANNOVAR**: Wang et al. (2010), Nucleic Acids Research. https://doi.org/10.1093/nar/gkq603
- **GATK Best Practices**: Van der Auwera & O'Connor (2020), O'Reilly Media
- **Source pipeline**: https://github.com/Pavel-Kravchenko/snp-calling-pipeline

---

[← Previous: Variant Calling and SNP Analysis](01_variant_calling_and_snp_analysis.ipynb) | [Next: RNA-seq Analysis: From Reads to Differential Expression →](../03_RNA_seq_Analysis/01_rna_seq_analysis.ipynb)

---